# WideBind — Colab Training (D=3584, T4)
221.73M params, 32 layers, PartitionedEmbed/Head (K=32, S=6), GroupedCognitiveMirror (32 experts, k=8, K-space gate), GroupedMLP (32), adaptive gates.

K-space gate: per-token per-expert sigmoid from hp + β·pred_error (k=8). w_gate: (G,d)→(G,k). gate_pred_scale learns W_pred coupling.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/widebind_data'
print(f'Data path: {DRIVE}')
!ls -lh "$DRIVE"/*.bin 2>/dev/null || echo 'No .bin files found'

In [ ]:
# Locate source files on Drive and copy locally
import sys, os, shutil
SRC_LOCAL = '/content/widebind_src'

src_drive = None
for d in [os.path.join(DRIVE, 'src'), DRIVE]:
    subdirs = ['core', 'compression', 'scripts', 'notebooks']
    if all(os.path.isdir(os.path.join(d, sd)) for sd in subdirs):
        src_drive = d
        break

if src_drive:
    if os.path.isdir(SRC_LOCAL):
        shutil.rmtree(SRC_LOCAL)
    shutil.copytree(src_drive, SRC_LOCAL)
    sys.path.insert(0, SRC_LOCAL)
    sys.path.insert(0, os.path.join(SRC_LOCAL, 'scripts'))
    print(f'Source files copied from {src_drive}')
else:
    print('Directories not found on Drive. Checking files...')
    for d in [os.path.join(DRIVE, 'src'), DRIVE]:
        for sd in ['core', 'compression', 'scripts']:
            p = os.path.join(d, sd)
            exists = os.path.isdir(p)
            print(f'  {p}: {"OK" if exists else "MISSING"}')
    print('\nUpload directories: core/, compression/, scripts/, notebooks/ to Drive')

%cd /content

In [ ]:
# Verify imports
from core import WideBindConfig, WideBindStack, LiveInference, MirrorMonitor
from compression import FCF_CPR

cfg = WideBindConfig(D=3584, n_layers=32, bind_K=32,
                     mlp_groups=32, mlp_expand=8,
                     code_dim=32, code_sparsity=6, mirror_k=8)
model = WideBindStack(cfg)
print(f'Model: {model.param_count():,} params')

# Quick sanity: check w_gate shape
wg = model.layers[0].mirror.w_gate
assert wg.shape == (32, 8), f'w_gate shape {wg.shape} != (32,8) -- old code!'
print(f'Gate: w_gate {list(wg.shape)}, gate_pred_scale={model.layers[0].mirror.gate_pred_scale.item():.4f}')
del model

In [ ]:
# ---- Patch FCF_CPR: sparse_block_codes instead of zeckendorf_codes ----
# Old fcf_cpr.py on Drive uses zeckendorf_codes(K~23); new model uses
# PartitionedEmbed/Head with K=32 sparse_block_codes.
from core import sparse_block_codes, dct_basis
from compression.fcf_cpr import FCF_CPR, dequantize_tensor

def _patched_decompress(self, compressed, meta, cfg):
    import torch
    sd = {}
    V_dct = dct_basis(cfg.D)
    codes = sparse_block_codes(cfg.vocab, K=cfg.code_dim, S=cfg.code_sparsity)
    n_layers = cfg.n_layers
    for k, v in compressed.items():
        info = meta.get(k)
        if info is None:
            sd[k] = v.clone()
        elif info[0] == 'scalar':
            orig_shape, orig_dtype = info[1], info[2]
            if len(orig_shape) > 0 and orig_shape[0] > 1:
                sd[k] = v.detach().expand(orig_shape).clone().to(orig_dtype)
            else:
                sd[k] = v.detach().clone().to(orig_dtype)
        elif info[0] == 'uniform8':
            shape, dtype, t_min, scale = info[1], info[2], info[3], info[4]
            sd[k] = dequantize_tensor(v, t_min, scale, dtype)
    for i in range(n_layers):
        sd[f'layers.{i}.V_dct'] = V_dct.clone()
    sd['embed.codes'] = codes.clone()
    sd['lm_head.codes'] = codes.clone()
    return sd

FCF_CPR.decompress_sd = _patched_decompress
print('FCF_CPR patched: decompress_sd uses sparse_block_codes')


In [ ]:
# ---- Launch training ----
# Config
D            = 3584
N_LAYERS     = 32
BIND_K       = 32
MLP_GROUPS   = 32
MLP_EXPAND   = 8
CODE_DIM     = 32
CODE_SPARSITY = 6
MIRROR_K     = 8
SEQ_LEN      = 128
LR           = 3e-4
MAX_STEPS    = 300000
WARMUP       = 2000
SAVE_EVERY   = 5000
EVAL_EVERY   = 1000
LOG_EVERY    = 100
SCHEDULER    = 'mirror'

print('=' * 60)
print(f'WideBind D={D} G={MLP_GROUPS}x{MLP_EXPAND}x')
print(f'  Layers: {N_LAYERS} | K={BIND_K} | codes={CODE_DIM}/{CODE_SPARSITY} | mirror_k={MIRROR_K}')
print(f'  LR={LR} | warmup={WARMUP} | steps={MAX_STEPS}')
print(f'  Scheduler: {SCHEDULER}')
print(f'  Save dir: {os.path.join(DRIVE, "checkpoints")}')
print('=' * 60)

In [ ]:
# ---- Training loop ----
import os, math, sys, time, glob
import torch
import torch.nn.functional as F
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {mem_total:.1f} GB')

# --- config (all 35+ fields from WideBindConfig; new params listed below) ---
cfg = WideBindConfig(
    # Architecture
    D=D, n_layers=N_LAYERS, bind_K=BIND_K,
    mlp_groups=MLP_GROUPS, mlp_expand=MLP_EXPAND,
    code_dim=CODE_DIM, code_sparsity=CODE_SPARSITY, mirror_k=MIRROR_K,
    seq_len=SEQ_LEN, lr=LR, max_steps=MAX_STEPS,
    warmup_steps=WARMUP, scheduler=SCHEDULER,
    data_dir=DRIVE, save_dir=os.path.join(DRIVE, 'checkpoints'),
    log_dir=os.path.join(DRIVE, 'logs'),
    grad_clip=1.0, conv_kernel=48,
    # --- New config-driven params (defaults shown) ---
    # w_pred_scale_init=0.5,        # coupling strength: pred_error ~14% of hp
    # gate_pred_scale_init=-1.0,     # init β for gate_pred_scale
    # gate_lr_mult=5.0,             # W_pred + w_pred_scale get LR×5
    # gate_pred_scale_mult=10.0,    # gate_pred_scale gets LR×10
    # w_d_init_std=0.1, conv_init_std=0.01,
    # spec_lo=0.5, spec_hi=1.5,
    # target_var=0.1, mag_threshold=0.3,
    # lr_min_ratio=0.05, max_decay_steps=50000,
    # var_min_for_lr_decay=0.001,
    # exploration_threshold=0.25, differentiation_threshold=0.08,
    # w_mem2v_scale_min=0.5, w_mem2v_scale_max=1.0,
    # ema_alpha_min=0.9, ema_alpha_max=0.99,
    # delta_var_ema_min=0.8, delta_var_ema_max=0.99,
    # noise_scale_min=0.001, noise_scale_max=0.05,
    # log_interval=100, eval_interval=1000, save_interval=5000,
)

# --- data ---
print(f'Loading data from {DRIVE}')
stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*.bin')))
if not stream_files:
    raise FileNotFoundError(f'No token_stream_*.bin in {DRIVE}')
streams = []
for f in stream_files:
    data = np.fromfile(f, dtype=np.uint16)
    streams.append(data)
total_tokens = sum(len(s) for s in streams)
print(f'Found {len(streams)} files, {total_tokens:,} total tokens')

# --- model ---
model = WideBindStack(cfg).to(device)
n_params = model.param_count()
print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')

# --- batch size ---
print('Finding optimal batch size...')
for bs in [8, 6, 4, 2, 1]:
    torch.cuda.empty_cache()
    try:
        x = torch.randint(0, 50000, (bs, SEQ_LEN), device=device)
        h = model.embed_tokens(x)
        out, state, _ = model(h)
        out[:, :1].sum().backward()
        model.zero_grad()
        print(f'  Batch size {bs}: OK')
        cfg.batch_size = bs
        break
    except RuntimeError:
        model.zero_grad()
        torch.cuda.empty_cache()
        print(f'  Batch size {bs}: OOM')
print(f'  Using batch size: {cfg.batch_size}')

# --- optimizer ---
from core import MirrorLRScheduler, AdaptiveController
param_groups = model.param_groups()
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(model, optimizer, cfg=cfg)

# --- FCF-CPR ---
cpr = FCF_CPR()

# --- resume ---
start_step = 0
best_val_loss = float('inf')
os.makedirs(cfg.save_dir, exist_ok=True)

def step_key(p):
    name = os.path.basename(p)
    # best.pt → 999999 (always newest), step_N.pt → N, interrupt_step_N.pt → N
    if name == 'best.pt':
        return 999999
    import re
    m = re.search(r'step_(\d+)', name)
    return int(m.group(1)) if m else 0
ckpt_files = sorted(glob.glob(os.path.join(cfg.save_dir, '*.pt')), key=step_key)
if ckpt_files:
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = cpr.load_compressed(latest, cfg=cfg)
    # --- Migration: K-space gate (w_gate shape changed: (G,d) -> (G,k)) ---
    # Old checkpoint has w_gate (32,112), new model expects (32,8)
    # Old checkpoint lacks gate_pred_scale entirely
    state_dict = model.state_dict()
    migrated = 0
    for k in list(ckpt['model'].keys()):
        if k in state_dict and ckpt['model'][k].shape != state_dict[k].shape:
            print(f'  Migrating {k}: {list(ckpt["model"][k].shape)} -> {list(state_dict[k].shape)} (reinit)')
            del ckpt['model'][k]
            migrated += 1
    if migrated:
        print(f'  Migrated {migrated} params from old checkpoint (gate rewrite)')
    # --- end migration ---
    missing, unexpected = model.load_state_dict(ckpt['model'], strict=False)
    if missing:
        gate_missing = [k for k in missing if 'gate' in k]
        mirror_missing = [k for k in missing if k not in gate_missing and ('mirror' in k or 'skip_alpha' in k or 'mod_' in k)]
        other_missing = [k for k in missing if k not in gate_missing and k not in mirror_missing]
        if gate_missing:
            print(f'  Gate params fresh init: {len(gate_missing)} ({gate_missing[0]}...)')
        if mirror_missing:
            print(f'  Per-expert mirror params fresh init: {len(mirror_missing)} (skip_alpha, mod_scale/bias)')
        if other_missing:
            print(f'  Other missing keys: {len(other_missing)} (trace/state buffers, expected)')
    if unexpected:
        print(f'  Unexpected keys from checkpoint: {len(unexpected)}')
    # --- Reinit migrated params from old checkpoints ---
    reinit_gate = 0
    for k, p in model.named_parameters():
        if 'gate_pred_scale' in k and p.item() < -2.0:
            p.data.fill_(-1.0)
            reinit_gate += 1
    if reinit_gate:
        print(f'  gate_pred_scale reinit: {reinit_gate} layers (→ -1.0)')
    # w_pred_scale: old init=0.1 per element, new default=0.5 — upgrade old checkpoints
    reinit_wps = 0
    for k, p in model.named_parameters():
        if 'w_pred_scale' in k and p.mean().item() < 0.15:
            p.data.fill_(0.5)
            reinit_wps += 1
    if reinit_wps:
        print(f'  w_pred_scale reinit: {reinit_wps} layers (0.1→0.5 — coupling boost)')
    # --- end reinit ---
    if 'optimizer' in ckpt:
        try:
            optimizer.load_state_dict(ckpt['optimizer'])
        except (ValueError, RuntimeError) as e:
            print(f'  WARNING: optimizer group mismatch ({e}). Starting fresh optimizer.')
    else:
        print('  WARNING: no optimizer in checkpoint, starting fresh')
    if 'scheduler' in ckpt:
        try:
            scheduler.load_state_dict(ckpt['scheduler'])
        except Exception as e:
            print(f'  WARNING: scheduler load failed ({e}), using step')
            scheduler._step = ckpt.get('step', 0)
    else:
        scheduler._step = ckpt.get('step', 0)
    start_step = ckpt['step']
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resume at step {start_step}')

# --- Monitor after resume ---
live = LiveInference(model, cfg, monitor=True) if ckpt_files else None

# --- helpers ---
@torch.no_grad()
def evaluate_model(model, streams, cfg, device):
    total_loss = 0.0
    total_steps = 0
    n_eval = min(100, sum(len(s) for s in streams) // (cfg.batch_size * cfg.seq_len) // len(streams))
    n_warmup = 10
    for stream in streams:
        offset = max(len(stream) // 4, cfg.batch_size * cfg.seq_len + 1)
        state = None
        gs = None
        for i in range(n_warmup):
            needed = cfg.batch_size * cfg.seq_len + 1
            if offset + needed > len(stream):
                offset = 0; break
            chunk = stream[offset:offset + needed]
            x = torch.from_numpy(chunk[:cfg.batch_size*cfg.seq_len].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            offset += cfg.batch_size * cfg.seq_len
            h = model.embed_tokens(x)
            out, state, gs = model(h, state, global_state=gs)
        for i in range(n_eval):
            needed = cfg.batch_size * cfg.seq_len + 1
            if offset + needed > len(stream):
                offset = 0; break
            chunk = stream[offset:offset + needed]
            x = torch.from_numpy(chunk[:cfg.batch_size*cfg.seq_len].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            y = torch.from_numpy(chunk[1:cfg.batch_size*cfg.seq_len+1].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            offset += cfg.batch_size * cfg.seq_len
            h = model.embed_tokens(x)
            out, state, gs = model(h, state, global_state=gs)
            loss = model.compute_loss(out, y)
            total_loss += loss.item()
            total_steps += 1
            if i % 50 == 49:
                state = None; gs = None
    return total_loss / max(total_steps, 1)

# --- train loop ---
state = None
global_state = None
stream_idx = 0
offset = 0
tokens_seen = 0
t0 = time.time()

print(f'Training: step {start_step} -> {MAX_STEPS}')
try:
    for step in range(start_step, MAX_STEPS):
        model.train()
        stream = streams[stream_idx]
        needed = cfg.batch_size * SEQ_LEN + 1
        if offset + needed > len(stream):
            offset = 0
            stream_idx = (stream_idx + 1) % len(streams)
            stream = streams[stream_idx]
            state = None; global_state = None
        chunk = stream[offset:offset + needed]
        x = torch.from_numpy(chunk[:cfg.batch_size*SEQ_LEN].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
        y = torch.from_numpy(chunk[1:cfg.batch_size*SEQ_LEN+1].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
        offset += cfg.batch_size * SEQ_LEN

        h = model.embed_tokens(x)
        out, state, global_state = model(h, state, global_state=global_state)
        loss = model.compute_loss(out, y)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  WARN: NaN/Inf loss at step {step}, skipping')
            optimizer.zero_grad(set_to_none=True)
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        tokens_seen += cfg.batch_size * SEQ_LEN
        current_lr = scheduler.get_last_lr()[0]

        if live is not None and step % LOG_EVERY == 0:
            live.monitor.capture(global_state)
            live.monitor.history['step'][-1] = step

        if step % LOG_EVERY == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(dt, 1e-8)
            mem_u = torch.cuda.max_memory_allocated() / 1e9
            log = f'  step={step:>6} loss={loss.item():.4f} lr={current_lr:.2e} '
            log += f'tok/s={tok_s:.0f} vram={mem_u:.1f}/{mem_total:.0f} GB'
            if live is not None:
                s = live.monitor.summary(window=100)
                expl = s.get('exploration', (0,))[0]
                mag = s.get('mirror_mag', torch.zeros(1))[0].item()
                gps0 = model.layers[0].mirror.gate_pred_scale.item()
                gpsL = model.layers[-1].mirror.gate_pred_scale.item()
                skip = model.layers[0].mirror.log_skip_alpha.exp().item()
                log += f' |mir|={mag:.4f} expl={expl:.3f} β0={gps0:.3f} βL={gpsL:.3f} α_skip={skip:.3f}'
            print(log)
            torch.cuda.reset_peak_memory_stats()

        if step > 0 and step % EVAL_EVERY == 0:
            val_loss = evaluate_model(model, streams, cfg, device)
            print(f'  EVAL step={step}: val_loss={val_loss:.4f} '
                  f'val_ppl={math.exp(val_loss):.2f}')
            torch.cuda.empty_cache()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                cpr.save_compressed({
                    'step': step, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_val_loss': best_val_loss, 'cfg': cfg,
                }, os.path.join(cfg.save_dir, 'best.pt'))
                print(f'  New best! Saved to best.pt (compressed)')

        if step > 0 and step % SAVE_EVERY == 0:
            save_path = os.path.join(cfg.save_dir, f'step_{step}.pt')
            cpr.save_compressed({
                'step': step, 'model': model.state_dict(),
                'best_val_loss': best_val_loss, 'cfg': cfg,
            }, save_path)
            print(f'  Checkpoint saved: step_{step}.pt (compressed)')

except KeyboardInterrupt:
    save_path = os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt')
    cpr.save_compressed({
        'step': step, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'cfg': cfg,
    }, save_path)
    print(f'\nInterrupted. Saved to {save_path} (compressed)')

In [ ]:
# --- Analyze latest checkpoint ---
from scripts.analyze_checkpoint import generate_report
import glob, shutil

latest_ckpt = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')))
if latest_ckpt:
    report_path = generate_report(latest_ckpt[-1])
    drive_report = os.path.join(cfg.log_dir, os.path.basename(report_path))
    os.makedirs(cfg.log_dir, exist_ok=True)
    shutil.copy2(report_path, drive_report)
    print(f'Report copied to {drive_report}')
    from IPython.display import HTML, display
    with open(report_path, 'r') as f:
        display(HTML(f.read()))
else:
    print('No checkpoints found')